# **DLO-JZ Data Augmentation**
<img src="./images/noun-car-repair-32305.png" style="float: left; margin-right: 1em;"/>

<div class="alert alert-block alert-success">

## Objet des notebooks

Dans ce TP, on se focalise sur les **problématiques de performance liées à la *Data Augmentation***. 

Les opérations de transformation des données d'entrée se font en général sur le CPU. On verra dans ce TP qu'il est possible de les déléguer au GPU si les ressources de calcul offertes par le CPU ne sont pas suffisantes.

Ce TP est divisé en trois parties, correspondant à trois types d'augmentation de données à implémenter :

* **TP 2.1** : RandAugment sur CPU
* **TP 2.2** : mixup sur CPU et GPU
* **TP 2.3** : CutMix sur GPU


Les cellules dans ce *notebook* ne sont pas prévues pour être modifiées, sauf rares exceptions indiquées dans les commentaires. Les TP se feront en modifiant les codes `dlojz_da_X.py`.

Les directives de modification seront marquées par l'étiquette suivante <div class="alert alert-block alert-warning">**TODO**</div>
Des solutions sont présentes dans le répertoire `solutions/`.

*Notebook rédigé par l'équipe assistance IA de l'IDRIS, juin 2026*

</div>

### **Environnement de calcul**

Les fonctions *python* de gestion de queue SLURM dévelopées par l'IDRIS et les fonctions dédiées à la formation DLO-JZ sont à importer.

Le module d'environnement pour les *jobs* et la taille des images sont fixés pour ce *notebook*.

<div class="alert alert-block alert-warning">
    
**TODO :** choisir un pseudonyme (maximum 5 caractères) pour vous différencier dans la queue SLURM pendant la formation.

</div>

In [ ]:
from idr_pytools import display_slurm_queue, gpu_jobs_submitter, search_log
from dlojz_tools import controle_technique, compare, GPU_underthehood, plot_accuracy, lrfind_plot, imagenet_starter, turbo_profiler
MODULE = 'pytorch-gpu/py3/2.8.0'
account = 'for@a100'
name = 'pseudo'   ## Pseudonyme à choisir

### **Gestion de la queue SLURM**

Pour afficher vos jobs dans la queue SLURM :

In [ ]:
display_slurm_queue(name)

**Remarque**: Cette fonction est utilisée plusieurs fois dans ce *notebook*. Elle permet d'afficher la queue de manière dynamique, rafraichie toutes les 5 secondes. Elle ne s'arrête que lorsque la queue est vide. Si vous désirez reprendre la main sur le *notebook*, il vous suffira d'arrêter manuellement la cellule avec le bouton *stop*. Cela a bien sûr aucun impact les jobs soumis.

Si vous voulez retirer TOUS vos jobs de la queue SLURM, décommenter et exécuter la cellule suivante :

In [ ]:
#!scancel -u $USER

Si vous voulez retirer UN de vos jobs de la queue SLURM, décommenter, compléter et exécuter la cellule suivante :

In [ ]:
#!scancel <jobid>

--------------

### Différence entre deux scripts
Pour comparer son code avec les solutions mises à disposition, la fonction suivante permet d'afficher une page html contenant un différentiel de fichiers texte.

In [ ]:
s1 = "dlojz_da_1.py"
s2 = "./solutions/dlojz_da_1.py"
compare(s1, s2)

Voir le résultat du différentiel de fichiers sur la page suivante (attention au spoil !) :

[compare.html](compare.html)

----------------------------------

<div class="alert alert-block alert-success">

## TP Data Augmentation 1 : RandAugment

Le but de ce TP est d'ajouter la transformation `RandAugment` (disponible dans *torchvision*) dans la liste des transformations pour la *Data Augmentation* et de mesurer son impact sur la performance du code.

Voir la [documentation torchvision sur RandAugment](https://pytorch.org/vision/stable/generated/torchvision.transforms.RandAugment.html).

</div>

## Garage - Mise à niveau
On fixe la *batch size* et la taille d'image pour ce TP.

In [ ]:
image_size = 224
bs_optim = 512

<div class="alert alert-block alert-info">

### Visualisation de RandAugment

Vous pouvez exécuter les cellules suivantes pour observer l'effet de la transformation `RandAugment` :

</div>

In [ ]:
import os
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import torch
import numpy as np
import matplotlib.pyplot as plt

transform = transforms.Compose([ 
        transforms.RandomResizedCrop(image_size),  # Random resize - Data Augmentation
        transforms.RandomHorizontalFlip(),  # Horizontal Flip - Data Augmentation
        transforms.RandAugment(5, 9),       # Random Augmentation 5: n operations, 9 : magnitude 
        transforms.ToTensor()               # convert the PIL Image to a tensor
        ])
    
train_dataset = torchvision.datasets.ImageNet(root=os.environ['ALL_CCFRSCRATCH']+'/imagenet',
                                                  transform=transform)
print(train_dataset)

In [ ]:
torch.manual_seed(0)
train_loader = torch.utils.data.DataLoader(dataset=train_dataset,    
                                           batch_size=4,
                                           shuffle=True)
batch = next(iter(train_loader))
print('X train batch, shape: {}, data type: {}, Memory usage: {} bytes'
      .format(batch[0].shape, batch[0].dtype, batch[0].element_size()*batch[0].nelement()))
print('Y train batch, shape: {}, data type: {}, Memory usage: {} bytes'
      .format(batch[1].shape, batch[1].dtype, batch[1].element_size()*batch[1].nelement()))

fig, ax = plt.subplots(2,2)
for i in range(4):
    img = batch[0][i].numpy().transpose((1,2,0))
    ax[i//2][i%2].imshow(img)
    ax[i//2][i%2].axis('off')
fig.show()

<div class="alert alert-block alert-info">

### Transformation RandAugment sur CPU


<div class="alert alert-block alert-warning">
    
**TODO :** dans le script [dlojz_da_1.py](dlojz_da_1.py)
</br>
Rajouter la transformation `RandAugment` dans la liste des transformations des images pour le *training* avec le paramétrage suivant :
* **Nombre d'opérations** = 5
* **Magnitude** = 9.

</div>
</div>

## Soumission du job
<u>**Attention vous sollicitez les noeuds de calcul à ce moment-là**.</u>

Pour soumettre le job, veuillez basculer la cellule suivante du mode `Raw NBConvert` au mode `Code` (raccourci: touche `Y` avec la cellule sélectionnée)

In [ ]:
command = f'dlojz_da_1.py -b {bs_optim} --image-size {image_size} --test'
n_gpu = 1
jobid = gpu_jobs_submitter(command, n_gpu, MODULE, name=name,
                    account=account, time_max='00:10:00')
print(f'jobid = {jobid}')

Puis, rebasculer la cellule précédente en mode `Raw NBConvert`, afin d'eviter de relancer un job par erreur (raccourci: touche `R` avec la cellule sélectionnée).

In [ ]:
display_slurm_queue(name)

In [ ]:
controle_technique(jobid)

In [ ]:
turbo_profiler(jobid)

<div class="alert alert-block alert-danger">

Assurez-vous que tout se passe bien avant de continuer
    
</div>
<img src="./images/cedez.png" style="float: left; margin-right: 1em;"/>